# Σύγκριση των δύο μεθόδων feature selection

Ίδιο μοντέλο, ίδιο πρωτόκολλο, ίδια 1142 μόρια. Αλλάζει **μόνο** το σύνολο των descriptors.

| cutoff | Μοντέλο (από το tuning) | MI ∩ CatBoost | Elastic Net |
|---|---|---|---|
| 50% | ExtraTrees + CatBoost (soft average) | 243 | 318 |
| 20% | ExtraTrees | 232 | 346 |

Η σύγκριση στο **cross-validation** έχει ήδη γίνει στα `pick_best_model_process.csv` των τεσσάρων tuning runs. Εδώ απαντιέται το ερώτημα στα **external test sets**: με το ίδιο μοντέλο, τι αλλάζει αν αλλάξει η μέθοδος επιλογής χαρακτηριστικών;

**Βήμα 1.** 5-fold CV στα 1142 μόρια, μόνο για να προκύψει το κατώφλι απόφασης (Youden's J). Κανένα test set δεν φορτώνεται.

**Βήμα 2.** Εκπαίδευση σε **όλα** τα 1142 μόρια και εφαρμογή στα Test set 1 και 2, με το κατώφλι του Βήματος 1 αμετάβλητο.

Ίδια λογική με το `test_set_evaluation_final`. Οι γραμμές του MI ∩ CatBoost πρέπει να αναπαράγουν τα αποτελέσματά του, και λειτουργούν ως έλεγχος ορθότητας.

Χρόνος: περίπου 3–5 λεπτά.

In [1]:
# ====================== ΠΑΡΑΜΕΤΡΟΙ ΚΑΙ ΒΟΗΘΗΤΙΚΕΣ ΣΥΝΑΡΤΗΣΕΙΣ ======================
import os
import warnings
import numpy as np
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import RobustScaler
from sklearn.ensemble import ExtraTreesClassifier
from catboost import CatBoostClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.metrics import (roc_auc_score, matthews_corrcoef, balanced_accuracy_score,
                             accuracy_score, f1_score, precision_score, recall_score,
                             confusion_matrix, roc_curve)

warnings.filterwarnings('ignore')

RANDOM_STATE = 42
LABELS = ['label_cutoff_50%', 'label_cutoff_20%']
OUT = '../results/Feature_selection_comparison'
os.makedirs(OUT, exist_ok=True)

CV = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

# Τα μοντέλα που επιλέχθηκαν στο hyperparameter_tunning_new, ανά cutoff
MODEL_CHOICE = {
    'label_cutoff_50%': ['ExtraTrees', 'CatBoost'],
    'label_cutoff_20%': ['ExtraTrees'],
}


def tag(lab):
    return lab.replace('label_cutoff_', '').replace('%', '')


def model_name(lab):
    m = MODEL_CHOICE[lab]
    return ' + '.join(m) + (' (soft average)' if len(m) > 1 else '')


def build(name):
    """Ίδιες υπερπαράμετροι με το test_set_evaluation_final."""
    if name == 'ExtraTrees':
        est = ExtraTreesClassifier(n_estimators=300, max_features='log2',
                                   min_samples_split=5, class_weight='balanced',
                                   random_state=RANDOM_STATE, n_jobs=-1)
    elif name == 'CatBoost':
        est = CatBoostClassifier(depth=4, learning_rate=0.15, iterations=300,
                                 random_seed=RANDOM_STATE, verbose=0,
                                 auto_class_weights='Balanced', thread_count=-1)
    else:
        raise ValueError(f'άγνωστο μοντέλο: {name}')
    return Pipeline([('scaler', RobustScaler()), ('model', est)])


def feature_sets(lab):
    """Τα δύο σύνολα descriptors του σεναρίου."""
    _mi = pd.read_excel(f'../Data/final_dataset_{lab}.xlsx', sheet_name='Dataset', nrows=1).columns
    mi = [c for c in _mi if c not in ('SMILES', 'smile')]
    en = pd.read_csv(f'../Data/elasticnet_selected_cutoff{tag(lab)}.csv')['descriptor'].tolist()
    return {'MI ∩ CatBoost': mi, 'Elastic Net': en}


def training_data(lab):
    """Τα 1142 μόρια του Training sheet με έγκυρη ετικέτα."""
    tr = pd.read_csv(f'../Data/merged_df_clean_{lab}.csv')
    tr = tr[tr[lab].notna()].reset_index(drop=True)
    return tr, tr[lab].astype(int).to_numpy()


def test_data(lab, test_set, feats, medians):
    """Descriptors και ετικέτες ενός test set· τα NaN συμπληρώνονται με διαμέσους του training."""
    lb = (pd.read_excel('../Data/hob_data_set.xlsx', sheet_name=test_set)[['smile', lab]]
          .rename(columns={'smile': 'SMILES'}).drop_duplicates(subset='SMILES'))
    d = (pd.read_csv(f"../Data/{test_set.replace('Test set ', 'test')}_descriptors_RAW.csv")
         .drop_duplicates(subset='SMILES'))
    m = d.merge(lb, on='SMILES', how='inner')
    m = m[m[lab].notna()]
    X = m[feats].apply(pd.to_numeric, errors='coerce').fillna(medians)
    return X, m[lab].astype(int).to_numpy()


def youden(y, proba):
    """Το κατώφλι που μεγιστοποιεί το Youden's J, από out-of-fold πιθανότητες."""
    fpr, tpr, thr = roc_curve(y, proba)
    return float(thr[np.argmax(tpr - fpr)])


def scores(y, proba, thr):
    pred = (proba >= thr).astype(int)
    tn, fp, fn, tp = confusion_matrix(y, pred, labels=[0, 1]).ravel()
    return {
        'ROC-AUC': roc_auc_score(y, proba),
        'MCC': matthews_corrcoef(y, pred),
        'Balanced Acc': balanced_accuracy_score(y, pred),
        'Accuracy': accuracy_score(y, pred),
        'F1': f1_score(y, pred, zero_division=0),
        'Precision': precision_score(y, pred, zero_division=0),
        'Sensitivity': recall_score(y, pred),
        'Specificity': tn / (tn + fp) if (tn + fp) else np.nan,
        'TP': tp, 'FP': fp, 'TN': tn, 'FN': fn,
    }


for _lab in LABELS:
    print(f"{_lab}: {model_name(_lab)}  |  "
          + ", ".join(f"{k} = {len(v)}" for k, v in feature_sets(_lab).items()))
print('\nΈτοιμο.')

label_cutoff_50%: ExtraTrees + CatBoost (soft average)  |  MI ∩ CatBoost = 243, Elastic Net = 318
label_cutoff_20%: ExtraTrees  |  MI ∩ CatBoost = 232, Elastic Net = 346

Έτοιμο.


## Βήμα 1 — Κατώφλι απόφασης από τα training δεδομένα

In [2]:
# ============ ΒΗΜΑ 1 — OUT-OF-FOLD ΚΑΙ ΚΑΤΩΦΛΙ (κανένα test set) ============
# Για κάθε cutoff και κάθε feature set: out-of-fold πιθανότητες του τελικού
# μοντέλου με 5-fold CV, και από αυτές το κατώφλι (Youden's J).
import time

cv_rows, OOF = [], {}
t0 = time.time()

for lab in LABELS:
    tr, y = training_data(lab)
    members = MODEL_CHOICE[lab]
    print(f"\n=== {lab} — {model_name(lab)} — {len(y)} μόρια "
          f"(High={int(y.sum())}, Low={int((y == 0).sum())}) ===")
    for fs, feats in feature_sets(lab).items():
        X = tr[feats]
        proba = np.mean([cross_val_predict(build(n), X, y, cv=CV,
                                           method='predict_proba', n_jobs=1)[:, 1]
                         for n in members], axis=0)
        thr = youden(y, proba)
        OOF[(lab, fs)] = (proba, thr)
        row = {'Label': lab, 'Feature set': fs, 'N features': len(feats),
               'Model': model_name(lab), 'Σύνολο': 'CV (1142)',
               'N molecules': len(y), 'Threshold (Youden)': round(thr, 3),
               **scores(y, proba, thr)}
        cv_rows.append(row)
        print(f"  {fs:14s} ({len(feats):3d} features): OOF AUC={row['ROC-AUC']:.4f}  "
              f"MCC={row['MCC']:.3f}  κατώφλι={thr:.3f}", flush=True)

cv = pd.DataFrame(cv_rows)
print(f"\nΧρόνος: {(time.time() - t0) / 60:.1f} λεπτά")


=== label_cutoff_50% — ExtraTrees + CatBoost (soft average) — 1142 μόρια (High=621, Low=521) ===
  MI ∩ CatBoost  (243 features): OOF AUC=0.7701  MCC=0.398  κατώφλι=0.572
  Elastic Net    (318 features): OOF AUC=0.7557  MCC=0.404  κατώφλι=0.553

=== label_cutoff_20% — ExtraTrees — 1142 μόρια (High=859, Low=283) ===
  MI ∩ CatBoost  (232 features): OOF AUC=0.7746  MCC=0.385  κατώφλι=0.713
  Elastic Net    (346 features): OOF AUC=0.7706  MCC=0.406  κατώφλι=0.654

Χρόνος: 0.5 λεπτά


## Βήμα 2 — Αξιολόγηση στα external test sets

In [3]:
# ============ ΒΗΜΑ 2 — TEST SET 1 ΚΑΙ 2 ============
# Εκπαίδευση σε ΟΛΑ τα 1142 μόρια και εφαρμογή με το κατώφλι του Βήματος 1.
# Τα test sets εμφανίζονται εδώ για πρώτη φορά.
test_rows = []

for lab in LABELS:
    tr, y = training_data(lab)
    members = MODEL_CHOICE[lab]
    for fs, feats in feature_sets(lab).items():
        X = tr[feats]
        medians = X.median()
        fitted = [build(n).fit(X, y) for n in members]
        thr = OOF[(lab, fs)][1]
        for ts in ['Test set 1', 'Test set 2']:
            X_te, y_te = test_data(lab, ts, feats, medians)
            proba = np.mean([m.predict_proba(X_te)[:, 1] for m in fitted], axis=0)
            row = {'Label': lab, 'Feature set': fs, 'N features': len(feats),
                   'Model': model_name(lab), 'Σύνολο': ts, 'N molecules': len(y_te),
                   'Threshold (Youden)': round(thr, 3), **scores(y_te, proba, thr)}
            test_rows.append(row)
            print(f"{lab} | {fs:14s} | {ts} (n={len(y_te)}): "
                  f"AUC={row['ROC-AUC']:.4f}  MCC={row['MCC']:.3f}  "
                  f"BA={row['Balanced Acc']:.3f}", flush=True)

test = pd.DataFrame(test_rows)

label_cutoff_50% | MI ∩ CatBoost  | Test set 1 (n=290): AUC=0.8202  MCC=0.478  BA=0.742
label_cutoff_50% | MI ∩ CatBoost  | Test set 2 (n=140): AUC=0.8947  MCC=0.629  BA=0.822
label_cutoff_50% | Elastic Net    | Test set 1 (n=290): AUC=0.8312  MCC=0.530  BA=0.769
label_cutoff_50% | Elastic Net    | Test set 2 (n=140): AUC=0.9262  MCC=0.650  BA=0.829
label_cutoff_20% | MI ∩ CatBoost  | Test set 1 (n=287): AUC=0.8301  MCC=0.456  BA=0.755
label_cutoff_20% | MI ∩ CatBoost  | Test set 2 (n=132): AUC=0.9811  MCC=0.366  BA=0.902
label_cutoff_20% | Elastic Net    | Test set 1 (n=287): AUC=0.8277  MCC=0.506  BA=0.759
label_cutoff_20% | Elastic Net    | Test set 2 (n=132): AUC=0.9795  MCC=0.456  BA=0.937


## Συγκεντρωτικοί πίνακες

In [4]:
# ====================== ΣΥΓΚΕΝΤΡΩΤΙΚΟΙ ΠΙΝΑΚΕΣ ======================
all_rows = pd.concat([cv, test], ignore_index=True)
all_rows['Σύνολο'] = pd.Categorical(all_rows['Σύνολο'],
                                    categories=['CV (1142)', 'Test set 1', 'Test set 2'],
                                    ordered=True)
all_rows = all_rows.sort_values(['Label', 'Σύνολο', 'Feature set']).reset_index(drop=True)

cols = ['Label', 'Model', 'Σύνολο', 'Feature set', 'N features', 'N molecules',
        'Threshold (Youden)', 'ROC-AUC', 'MCC', 'Balanced Acc', 'Accuracy',
        'F1', 'Sensitivity', 'Specificity']
print('=== Αναλυτικά ===')
print(all_rows[cols].round(4).to_string(index=False))

print('\n=== Διαφορά MI ∩ CatBoost − Elastic Net ===')
for metric in ['ROC-AUC', 'MCC', 'Balanced Acc']:
    p = all_rows.pivot_table(index=['Label', 'Model', 'Σύνολο'], columns='Feature set',
                             values=metric, observed=False)
    p['Διαφορά'] = (p['MI ∩ CatBoost'] - p['Elastic Net']).round(4)
    print(f'\n-- {metric} --')
    print(p.round(4).to_string())

# --------------------------------------------------------------- αποθήκευση
all_rows[cols + ['TP', 'FP', 'TN', 'FN']].round(4).to_csv(
    f'{OUT}/feature_selection_comparison.csv', index=False, encoding='utf-8-sig')

with pd.ExcelWriter(f'{OUT}/feature_selection_comparison.xlsx') as w:
    all_rows[cols + ['TP', 'FP', 'TN', 'FN']].round(4).to_excel(
        w, sheet_name='Αναλυτικά', index=False)
    for metric in ['ROC-AUC', 'MCC', 'Balanced Acc', 'F1']:
        (all_rows.pivot_table(index=['Label', 'Model', 'Σύνολο'], columns='Feature set',
                              values=metric, observed=False).round(4)
         .to_excel(w, sheet_name=metric.replace('-', '').replace(' ', '')[:31]))

print(f"\nΑποθηκεύτηκαν:\n  {OUT}/feature_selection_comparison.csv"
      f"\n  {OUT}/feature_selection_comparison.xlsx")

=== Αναλυτικά ===
           Label                                Model     Σύνολο   Feature set  N features  N molecules  Threshold (Youden)  ROC-AUC    MCC  Balanced Acc  Accuracy     F1  Sensitivity  Specificity
label_cutoff_20%                           ExtraTrees  CV (1142)   Elastic Net         346         1142               0.654   0.7706 0.4062        0.7041    0.7776 0.8518       0.8498       0.5583
label_cutoff_20%                           ExtraTrees  CV (1142) MI ∩ CatBoost         232         1142               0.713   0.7746 0.3852        0.7159    0.7242 0.7997       0.7322       0.6996
label_cutoff_20%                           ExtraTrees Test set 1   Elastic Net         346          287               0.654   0.8277 0.5064        0.7587    0.8084 0.8700       0.8598       0.6575
label_cutoff_20%                           ExtraTrees Test set 1 MI ∩ CatBoost         232          287               0.713   0.8301 0.4562        0.7551    0.7491 0.8154       0.7430       0.76